In [1]:
print('test')

test


In [ ]:
import pandas as pd
# 실행
food_df = pd.read_excel('./data/20251229_음식DB 19495건.xlsx')

In [ ]:

import numpy as np

def reduce_and_evaluate_food_data(food_df):
    # ---------------------------------------------------------
    # 1. 데이터 전처리 (컬럼 추출 및 형변환)
    # ---------------------------------------------------------
    
    # 분석용 핵심 컬럼 추출 (원본 보호를 위해 copy)
    core_columns = [
        '업체명', '식품명', '영양성분함량기준량',
        '에너지(kcal)', '단백질(g)', '지방(g)', '탄수화물(g)', 
    ]
    df_core = food_df[core_columns].copy()
    
    # 기준량 데이터 정제 (문자열에서 숫자만 추출 후 변환)
    df_core['영양성분함량기준량'] = df_core['영양성분함량기준량'].astype(str).str.extract(r'(\d+\.?\d*)')
    df_core['영양성분함량기준량'] = pd.to_numeric(df_core['영양성분함량기준량'], errors='coerce')
    
    # 영양성분 컬럼 숫자형 변환
    numeric_cols = [
        '에너지(kcal)', '단백질(g)', '지방(g)', '탄수화물(g)', 
    ]
    for col in numeric_cols:
        df_core[col] = pd.to_numeric(df_core[col], errors='coerce')
    
    # 결측치 -1 대체 및 컬럼명 직관화 (-1로 채워 실제 0과 구분)
    df_core.fillna(-1, inplace=True)
    df_core = df_core.rename(columns={'영양성분함량기준량': '기준량(g)'})
    
    # ---------------------------------------------------------
    # 2. 영양학적 지표 계산 (분모나 분자가 -1(결측)일 때는 결과도 -1로)
    # ---------------------------------------------------------
    
    # [2-1] 에너지 밀도 (1g당 열량)
    df_core['에너지밀도(kcal/g)'] = np.where(
        (df_core['기준량(g)'] > 0) & (df_core['에너지(kcal)'] != -1), 
        df_core['에너지(kcal)'] / df_core['기준량(g)'], 
        -1
    )
    
    # [2-2] 3대 영양소 에너지 비율 (%) (탄수화물 4kcal, 단백질 4kcal, 지방 9kcal 기준)
    df_core['탄수화물_에너지비율(%)'] = np.where((df_core['에너지(kcal)'] > 0) & (df_core['탄수화물(g)'] != -1), (df_core['탄수화물(g)'] * 4 / df_core['에너지(kcal)']) * 100, -1)
    df_core['단백질_에너지비율(%)'] = np.where((df_core['에너지(kcal)'] > 0) & (df_core['단백질(g)'] != -1), (df_core['단백질(g)'] * 4 / df_core['에너지(kcal)']) * 100, -1)
    df_core['지방_에너지비율(%)'] = np.where((df_core['에너지(kcal)'] > 0) & (df_core['지방(g)'] != -1), (df_core['지방(g)'] * 9 / df_core['에너지(kcal)']) * 100, -1)
    
    # [2-3] 세부 영양 품질 지표
    # 단백질 INQ (하루 권장량: 에너지 2,000kcal, 단백질 55g 기준)
    # 1.0 이상이면 단백질이 충분히 포함된 식품으로 간주
    df_core['단백질_INQ'] = np.where(
        (df_core['에너지(kcal)'] > 0) & (df_core['단백질(g)'] != -1),
        (df_core['단백질(g)'] / 55) / (df_core['에너지(kcal)'] / 2000),
        -1
    )
    
   
    
    
    # ---------------------------------------------------------
    # 3. 데이터 후처리
    # ---------------------------------------------------------
    
    # 무한대 값(inf) -1 치환 (오류성 값 처리)
    df_core.replace([np.inf, -np.inf], -1, inplace=True)
    
    return df_core.round(2)



food_cleaned_df = reduce_and_evaluate_food_data(food_df)
food_cleaned_df.head()

,업체명,식품명,기준량(g),에너지(kcal),단백질(g),지방(g),탄수화물(g)
0,해당없음,국밥_돼지머리,100,137,6.70,5.16,15.94
1,해당없음,국밥_순대국밥,100,75,3.17,2.28,10.38
2,해당없음,국밥_콩나물,100,52,1.45,0.24,10.93
3,해당없음,기장밥,100,166,3.44,0.57,36.77
4,해당없음,김밥,100,140,4.84,4.55,19.98


In [19]:
food_df_2 = food_cleaned_df[food_cleaned_df['업체명'] != '해당없음']
food_df_2.head()

food_df_2.to_csv('./data/cleaned_food_data.csv', index=False, encoding='utf-8-sig')